# NB02: Data Transformation

This notebook turns the raw data file saved in the notebook 1 into a tidy table, with the necessary variables extracted and prepared for the analysis in notebook 3.


`diff_winners = winner's winners vs loser's winners`


`diff_ue = loser's errors vs winner's errors`

## Setup

In [17]:
import pandas as pd
import json
import os


step 1 first we drop the empty rows as specified in `NB01-Data-Collection.ipynb`

In [18]:
file_path = '../data/raw/dump.json'

df = pd.read_json(file_path)

df_filtered1 = df[df["statistics"].apply(bool)].reset_index(drop=True)

print(df_filtered1)


     first_player_key  second_player_key   event_winner  tournament_name  \
0                 396               1742   First Player  Australian Open   
1                 424               9209  Second Player  Australian Open   
2               39913               1095   First Player  Australian Open   
3                3319               1082  Second Player  Australian Open   
4                7912               3696  Second Player  Australian Open   
..                ...                ...            ...              ...   
707               372               8174  Second Player        Wimbledon   
708              2832               1980  Second Player        Wimbledon   
709              2072               1905   First Player        Wimbledon   
710              8174               1980  Second Player        Wimbledon   
711              2072               1980   First Player        Wimbledon   

                                            statistics  
0    [{'player_key': 396, 'sta

step 2 now we fill into create the winner/loser columns

In [19]:
print(df_filtered1["event_winner"].unique())

df_filtered1a = df_filtered1[df_filtered1["event_winner"] == "First Player"]
df_filtered1b = df_filtered1[df_filtered1["event_winner"] == "Second Player"]

df_filtered1a["won"] = df_filtered1a["first_player_key"]
df_filtered1a["lost"] = df_filtered1a["second_player_key"]

df_filtered1b["won"] = df_filtered1b["second_player_key"]
df_filtered1b["lost"] = df_filtered1b["first_player_key"]

df_filtered2 = pd.concat([df_filtered1a, df_filtered1b], ignore_index=True)


<ArrowStringArray>
['First Player', 'Second Player']
Length: 2, dtype: str


now we outline statistics

In [20]:
df_filtered2["statistics"][0]

[{'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': 'Aces',
  'stat_value': '0',
  'stat_won': None,
  'stat_total': None},
 {'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': 'Double Faults',
  'stat_value': '2',
  'stat_won': None,
  'stat_total': None},
 {'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': '1st serve percentage',
  'stat_value': '64%',
  'stat_won': None,
  'stat_total': None},
 {'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': '1st serve points won',
  'stat_value': '68%',
  'stat_won': 30,
  'stat_total': 44},
 {'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': '2nd serve points won',
  'stat_value': '64%',
  'stat_won': 16,
  'stat_total': 25},
 {'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': 'Break Points Saved',
  'stat_value': '67%',
  'stat_won'

In [21]:
def extract_stats(nest):

    if type(nest) != list:
        return "ERROR: Not a list"

    kept = []
    keep = ["Winners", "Unforced errors", "Aces", "Double Faults"]
    
    for item in nest:
        if item["stat_period"] == "match":
            if item["stat_name"] in keep:
                kept.append(item)

    stat_dict = {}
    for entry in kept:
        if entry["player_key"] not in stat_dict.keys():
            stat_dict[entry["player_key"]] = {}
    
        stat_dict[entry["player_key"]][entry["stat_name"]] = entry["stat_value"]

    return stat_dict

In [22]:
df_filtered2["statistics"] = df_filtered2["statistics"].apply(extract_stats)

In [23]:
df_filtered2["statistics"][708]

{2073: {'Aces': '29',
  'Double Faults': '5',
  'Winners': '74',
  'Unforced errors': '61'},
 1905: {'Aces': '14',
  'Double Faults': '6',
  'Winners': '43',
  'Unforced errors': '47'}}

In [24]:
def process_stats(row):
    winner = row["won"]
    loser = row["lost"]
    stats = row["statistics"]
    
    to_return = {"winner_winners": int(stats[winner]["Winners"]),
                "winner_ue": int(stats[winner]["Unforced errors"]),
                "winner_aces": int(stats[winner]["Aces"]),
                "winner_df": int(stats[winner]["Double Faults"]),
                "loser_winners": int(stats[loser]["Winners"]),
                "loser_ue": int(stats[loser]["Unforced errors"]),  
                "loser_aces": int(stats[loser]["Aces"]), 
                "loser_df":int(stats[loser]["Double Faults"]), 
                }

    return pd.Series(to_return)

In [25]:
df_filtered2[["winner_winners", "winner_ue", "winner_aces", "winner_df", "loser_winners", "loser_ue", "loser_aces", "loser_df"]] = df_filtered2.apply(process_stats, axis=1)

df_filtered3 = df_filtered2.drop(columns = ["event_winner", "first_player_key", "second_player_key", "statistics", "won", "lost"])


In [26]:
df_filtered3


,tournament_name,winner_winners,winner_ue,winner_aces,winner_df,loser_winners,loser_ue,loser_aces,loser_df
0,Australian Open,14,19,0,2,22,35,6,4
1,Australian Open,16,7,4,0,14,27,3,3
2,Australian Open,37,28,6,2,21,31,1,4
3,Australian Open,20,20,2,1,9,29,1,3
4,Australian Open,8,13,1,1,11,29,2,2
...,...,...,...,...,...,...,...,...,...
707,Wimbledon,36,33,7,0,72,59,19,7
708,Wimbledon,43,47,14,6,74,61,29,5
709,Wimbledon,27,15,8,1,21,41,6,2
710,Wimbledon,29,15,14,2,31,28,17,2
